# Loop Stability: Nyquist Criterion and Robustness Wedge

This notebook demonstrates four examples from control-systems literature:

1. **Unconditionally stable** — First-order plant with proportional control. No phase crossover → infinite gain margin → no wedge is drawn.

2. **Conditionally stable (unstable for high gain)** — Third-order plant (integrator + two poles). For sufficiently high gain, the Nyquist plot encircles -1. The wedge highlights the danger zone.

3. **High phase margin with finite gain margin** — Triple-pole plant. PM > 90° at the gain crossover, so the wedge extends to the **right** of the critical point (θ ∈ [π−PM, π+PM] includes angles where cos θ > 0).

4. **Unstable open-loop plant** — From [Toronto Met textbook](https://pressbooks.library.torontomu.ca/controlsystems/chapter/14-7-solved-examples-of-nyquist-stability-criterion/): plant with RHP pole; closed loop stable for K>2 (one CW encirclement of -1 required).

The **robustness wedge** (red shaded region) is centered at -1 and encodes:
- **Angular extent**: ± phase margin (PM) around the negative real axis
- **Radial extent**: distances from gain-margin crossings on the real axis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from loopkit.loop import LOOP
from loopkit.component import Component
import loopkit.loopmath as lm

In [ ]:
sps = 10e3  # Sample rate (Hz)
frfr = np.logspace(-3, 2, 2000)  # Frequency array for Nyquist

## Example 1: Unconditionally Stable

**Open-loop**: $L(s) = \frac{K}{s+1}$ with $K=2$

First-order plant with proportional control. The Nyquist contour is a semicircle in the right half-plane; it never encircles -1 for any positive $K$. The wedge shows large phase and gain margins.

In [ ]:
# L(s) = K/(s+1), K=2
K_stable = 2.0
plant_stable = Component("Plant", sps, tf=f"{K_stable}/(s+1)", domain="s")
loop_stable = LOOP(sps, [plant_stable], name="Unconditionally stable")

ugf_stable, pm_stable = lm.get_margin(loop_stable.Gf(f=frfr), frfr, deg=True)
print(f"UGF = {ugf_stable:.4e} Hz")
print(f"Phase margin = {pm_stable:.2f} deg")

In [ ]:
ax = loop_stable.nyquist_plot(
    frfr, which="G", critical_point=True, unity_gain_circle=True,
    figsize=(5, 5), title="Unconditionally stable: L(s) = 2/(s+1)"
)
plt.tight_layout()
plt.show()

## Sensitivity keep-out zone

The most useful robustness overlay on the Nyquist plot is the **sensitivity keep-out zone**: a disk centered at -1 with radius \(1/M_s\), where \(M_s\) is the maximum allowed sensitivity peak.

- **Interpretation**: If the Nyquist locus enters this disk, the closed-loop sensitivity \(S = 1/(1+L)\) exceeds \(M_s\) at some frequency (disturbances and model error are amplified too much).
- **Staying outside** the disk guarantees \(\max |S| \leq M_s\).
- **Typical targets**: \(M_s \approx 1.4\)–\(2.0\) (1.4 = conservative, 2.0 = more aggressive).
- **Advantage over phase/gain margins**: Well-defined even with multiple crossings, nonminimum-phase shapes, or ambiguous crossovers.

In [ ]:
# Same unconditionally stable loop, now with sensitivity keep-out (M_s = 1.6)
# The Nyquist locus stays outside the disk → sensitivity peak ≤ 1.6
ax = loop_stable.nyquist_plot(
    frfr, which="G", critical_point=True, unity_gain_circle=True,
    sensitivity_keepout=1.6,
    figsize=(5, 5), title="Sensitivity keep-out: M_s ≤ 1.6"
)
plt.tight_layout()
plt.show()

In [ ]:
# Compare: loop that approaches -1 (triple pole, higher gain)
# With M_s=1.4 the locus may enter the disk → sensitivity peak > 1.4
loop_peaky = LOOP(sps, [Component("Plant", sps, tf="2.5/(s+1)**3", domain="s")], name="Peaky")
ax = loop_peaky.nyquist_plot(
    frfr, which="G", critical_point=True, unity_gain_circle=True,
    sensitivity_keepout=1.4,
    figsize=(5, 5), title="Sensitivity keep-out: locus enters disk (M_s > 1.4)"
)
ax.set_xlim(-2.5, 1.5)
ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

## Example 2: Conditionally Stable (Unstable for High Gain)

**Open-loop**: $L(s) = \frac{K}{s(s+1)(s+2)}$ with $K=8$

Third-order plant (integrator + two real poles). For low gain the closed loop is stable; for high gain the Nyquist plot encircles -1 clockwise, indicating instability. With $K=8$ the system is unstable (negative phase margin). The wedge shows the critical region around -1.

In [ ]:
# L(s) = K/(s*(s+1)*(s+2)) = K/(s^3 + 3*s^2 + 2*s)
K_unstable = 8.0
plant_unstable = Component(
    "Plant", sps,
    tf=f"{K_unstable}/(s*(s+1)*(s+2))",
    domain="s"
)
loop_unstable = LOOP(sps, [plant_unstable], name="Conditionally stable")

ugf_unstable, pm_unstable = lm.get_margin(loop_unstable.Gf(f=frfr), frfr, deg=True)
print(f"UGF = {ugf_unstable:.4e} Hz")
print(f"Phase margin = {pm_unstable:.2f} deg")

In [ ]:
ax = loop_unstable.nyquist_plot(
    frfr, which="G", critical_point=True, unity_gain_circle=True,
    sensitivity_keepout=1.6,
    figsize=(5, 5), title="Unstable: L(s) = 8/(s(s+1)(s+2))"
)
ax.set_xlim(-3, 2)
ax.set_ylim(-2.5, 2.5)
plt.tight_layout()
plt.show()

## Example 3: High Phase Margin with Finite Gain Margin

**Open-loop**: $L(s) = \frac{K}{(s+1)^3}$ with $K=1.5$

Third-order plant (triple pole at -1). Phase goes from 0° to -270°, crossing -180° at a finite frequency, so there is a finite gain margin. The gain crossover occurs while phase is still above -90°, giving PM > 90°. The wedge therefore extends to the **right** of the critical point (into the fourth quadrant), since θ ∈ [π−PM, π+PM] includes angles where cos(θ) > 0 when PM > 90°.

In [ ]:
# L(s) = K/(s+1)^3, K=1.5
K_highpm = 1.5
plant_highpm = Component(
    "Plant", sps,
    tf=f"{K_highpm}/(s+1)**3",
    domain="s"
)
loop_highpm = LOOP(sps, [plant_highpm], name="High PM, finite GM")

ugf_highpm, pm_highpm = lm.get_margin(loop_highpm.Gf(f=frfr), frfr, deg=True)
print(f"UGF = {ugf_highpm:.4e} Hz")
print(f"Phase margin = {pm_highpm:.2f} deg (PM > 90° → wedge extends right of -1)")

In [ ]:
ax = loop_highpm.nyquist_plot(
    frfr, which="G", critical_point=True, unity_gain_circle=True,
    sensitivity_keepout=1.6,
    figsize=(5, 5), title="High PM: L(s) = 1.5/(s+1)³"
)
# ax.set_xlim(-2, 1)
# ax.set_ylim(-1.5, 1.5)
plt.tight_layout()
plt.show()

## Example 4: Unstable Open-Loop Plant (from [Toronto Met](https://pressbooks.library.torontomu.ca/controlsystems/chapter/14-7-solved-examples-of-nyquist-stability-criterion/))

**Open-loop**: $L(s) = K\,\frac{s+2}{s(s-2)}$ with $K=3$

The plant has an **unstable pole** at $s=+2$ (P=1). With unity feedback, the closed loop is stable for $K>2$: the Nyquist plot must encircle -1 once clockwise. With $K=3$, the system is stable. This example illustrates the Nyquist criterion when the open loop has RHP poles.

In [ ]:
# L(s) = K*(s+2)/(s*(s-2)), K=3 — unstable open-loop (RHP pole at s=2)
K_unstable_ol = 3.0
plant_unstable_ol = Component(
    "Plant", sps,
    tf=f"{K_unstable_ol}*(s+2)/(s*(s-2))",
    domain="s"
)
loop_unstable_ol = LOOP(sps, [plant_unstable_ol], name="Unstable open-loop")

ugf_uol, pm_uol = lm.get_margin(loop_unstable_ol.Gf(f=frfr), frfr, deg=True)
print(f"UGF = {ugf_uol:.4e} Hz")
print(f"Phase margin = {pm_uol:.2f} deg")
print("Stable for K>2 (one CW encirclement of -1 required, P=1)")

In [ ]:
ax = loop_unstable_ol.nyquist_plot(
    frfr, which="G", critical_point=True, unity_gain_circle=True,
    sensitivity_keepout=1.6,
    figsize=(5, 5), title="Unstable open-loop: L(s) = 3(s+2)/(s(s-2)), stable for K>2"
)
ax.set_xlim(-3, 2)
ax.set_ylim(-2.5, 2.5)
plt.tight_layout()
plt.show()

## Summary

| Example | L(s) | Stability | Wedge interpretation |
|---------|------|-----------|----------------------|
| 1 | 2/(s+1) | Unconditionally stable | No wedge (infinite GM) |
| 2 | 8/(s(s+1)(s+2)) | Unstable (high gain) | Nyquist encircles -1; negative phase margin |
| 3 | 1.5/(s+1)³ | Stable, high PM | PM > 90° → wedge extends to the right of -1 |
| 4 | 3(s+2)/(s(s-2)) | Unstable open-loop, stable closed-loop | P=1, stable for K>2 |

The **stability-margin wedge** (red shaded region) indicates the zone around -1 that the Nyquist curve should avoid for robust stability. Entering the wedge implies reduced phase or gain margin.